# Sections 4.1–4.2: exact parameter recovery
For *Recurrence formulas for Taylor expansions at CM points on genus-one and genus-two Shimura curves*.
Choose the **SageMath** kernel, then **Restart Kernel and Run All**. This notebook
uses only Sage and has no companion-file dependencies.

**Inputs:** the models and differentials (equivalently $H$), the marked CM centers
$u_0$, Table 3's allowed families, and the stated initial identities $c_n$ from
Appendix A. For $14a,15a,21a$ we additionally use Tu's published
$Q(T)=\frac12\{z,T\}$ as quoted in §4.1. These geometric identifications and exact
coefficient recognitions are hypotheses; the recovery and uniqueness below are exact.

We derive $\mathcal R,L,u_1$ for all twelve forms. The marked $u_0$ is supplied
by the geometry, not recovered from numerical coefficients. Final printed answers
are used only in the last comparison, after solving.

In [1]:
from sage.all import *
from IPython.display import display, Math, Markdown

Pu = PolynomialRing(QQ, 'u'); u = Pu.gen(); Fu = Pu.fraction_field()
Pt = PolynomialRing(QQ, 't'); t = Pt.gen(); K = Pt.fraction_field()
PT = PolynomialRing(QQ, 'T'); T = PT.gen()

def rationals(s):
    return [QQ(x) for x in s.split()]

def at(f, x):
    return f.numerator()(x) / f.denominator()(x)

def schwarzian(f):
    return f.derivative(3)/f.derivative() - QQ(3)/2*(f.derivative(2)/f.derivative())**2

def monomial(p):
    return bool(p) and len(p.exponents()) == 1

recovered, fits = {}, {}

## Geometric data and assumed coefficients
Each row is $(H,u_0,[c_0,\ldots,c_m])$, with $c_0=c_1=1$ and
$u_t^2=H(u)$ for the specified differential. For the even quartics with
$u=x^2$, $H=4u\,G(u)$ when $y^2=G(x^2)$. The other $H$ values are the
quotient-coordinate expressions in Tables 1–2, derived in notebook 1.
All these polynomials and centers are included here so this notebook runs alone.

In [2]:
data = {
    "14a": (4*u*(-u**2+13*u-128), QQ("16"),
             rationals("1 1 -4 214/9")),
    "15a": (4*u*(-3*u**2-82*u-27), QQ("9"),
             rationals("1 1 -1/4 27/16")),
    "21a": (4*u*(-7*u**2+94*u-343), QQ("441/25"),
             rationals("1 1 -905/3969 -1318/27783")),
    "33a": (4*u*(-3*u**2-10*u-243), QQ("9/25"),
             rationals("1 1 -72337/529 9202302/12167 3009372468/279841 -1149714703920/6436343")),
    "34a": ((u**2+4)*(-3*u**2+26*u-59), QQ("-15/4"),
             rationals("1 1 -12/25 2762/125 -17736/625 -67648/125")),
    "46a": (4*u*(-u**2+45*u-512), QQ("3136/81"),
             rationals("1 1 27824049/123402050 -114816966372/484661551375 -784666539922896/3807016486050625 -3694325348311426944/5980822899585531875")),
    "26a": (4*(-2*u**3+19*u**2-24*u-169), QQ("1"),
             rationals("1 1 -16/147 274/3087 -2488/9261 352768/1361367")),
    "26b": (4*(-169*u**3-24*u**2+19*u-2), QQ("1"),
             rationals("1 1 40 -734 -11240 1308544")),
    "38a": (4*(-16*u**3-59*u**2-82*u-19), QQ("1"),
             rationals("1 1 -64/27 -62/27 -19672/2187 41984/6561")),
    "38b": (4*(-19*u**3-82*u**2-59*u-16), QQ("1"),
             rationals("1 1 -16/169 -878/2197 7688/28561 -2699648/371293")),
    "58a": (4*(-2*u**3-78*u**2-862*u-1682), QQ("-11"),
             rationals("1 1 -4/5 314/125 31192/625 -1977728/625 4428752/3125")),
    "58b": (4*(-1682*u**3-862*u**2-78*u-2), QQ("-1/11"),
             rationals("1 1 140/9 262/3 -39592/81 420224/81")),
}
assert len(data) == 12 and all(c[:2] == [1,1] and H(a) != 0 for H,a,c in data.values())

The jet formulas (4.2) and (4.13) can be verified once with symbolic
$c_2,c_3,c_4,c_5$. The division by $A^4$ is valid because $A(0)=1$.

In [3]:
Jbase = PolynomialRing(QQ, names=('c2','c3','c4','c5'))
c2,c3,c4,c5 = Jbase.gens()
Jseries = PowerSeriesRing(Jbase.fraction_field(), 'q', default_prec=6)
AA = Jseries([1,1,c2/2,c3/6,c4/24,c5/120]).add_bigoh(6)
JJ = (QQ(3)/2*AA.derivative()**2-AA*AA.derivative(2))/AA**4
assert [factorial(j)*JJ[j] for j in range(4)] == [
    QQ(3)/2-c2, 6*c2-c3-6, 6*c2**2+9*c3-c4-42*c2+30,
    21*c2*c3-108*c2**2+330*c2-78*c3+12*c4-c5-180]
print('PASS: all four symbolic reconstruction jets in (4.2) and (4.13).')

PASS: all four symbolic reconstruction jets in (4.2) and (4.13).


## §4.1: transport Tu's Schwarzian and recover $L,u_1$
For $T=\phi(u)$, the chain rule and $u_t^2=H(u)$ give
$$\mathcal R=2H\,Q(\phi)\phi'^2+H\{\phi,u\}
             +\tfrac12 H''-\frac{3H'^2}{8H}.$$
The source-to-target changes are $(x,y)=(4s,4v)$ at level 14 and
$(x,y)=(s,v)$ at levels 15 and 21. Equal horizontal and vertical scale
factors preserve $dx/y=ds/v$. We check the model identities before transport.
The first two coefficient identities then give
$$L=\frac{3/2-c_2}{\mathcal R(u_0)},\qquad
u_1=\frac{6c_2-c_3-6}{L\mathcal R'(u_0)}.$$
The nonzero denominators and $u_1^2=LH(u_0)$ are checked explicitly.

In [4]:
Q = {
    '14a': (192+440*T+43*T**2+1036*T**3+960*T**4)/(16*T**2*(8+13*T+16*T**2)**2),
    '15a': (177147-244944*T+244242*T**2-3680*T**3+35*T**4)/(144*T**2*(1-T)**2*(81-T)**2),
    '21a': 21*(40353607-17647350*T+3561369*T**2-477652*T**3+31833*T**4-630*T**5+7*T**6)
                /(16*T**2*(49-T)**2*(343-94*T+7*T**2)**2)}
# Source G(s^2), common x/y scale, and the resulting T=phi(u).
source = {'14a': (-16*u**2+13*u-8, 4, -u/16),
          '15a': (-(3*u+1)*(u+27), 1, -3*u),
          '21a': (-7*u**2+94*u-343, 1, u)}
for name, (G, scale, phi) in source.items():
    H, a, c = data[name]
    assert Pu(H/(4*u))(scale**2*u) == scale**2*G
    phi = Fu(phi)
    R = Fu(2*H*at(Q[name], phi)*phi.derivative()**2 + H*schwarzian(phi)
           + H.derivative(2)/2 - 3*H.derivative()**2/(8*H))
    assert R.denominator()(a) != 0 and at(R, a) != 0
    L = (QQ(3)/2-c[2])/at(R, a)
    assert L*at(R.derivative(), a) != 0
    b = (6*c[2]-c[3]-6)/(L*at(R.derivative(), a))
    assert b**2 == L*H(a)
    recovered[name] = (R, L, a, b)
    display(Math(r'\mathcal R_{'+name+r'}(u)='+latex(R)))
    print(name, ': L =', L, ', u0 =', a, ', u1 =', b)

<IPython.core.display.Math object>

14a : L = -11/9 , u0 = 16 , u1 = -352/3


<IPython.core.display.Math object>

15a : L = -7/64 , u0 = 9 , u1 = -63


<IPython.core.display.Math object>

21a : L = -43/3969 , u0 = 441/25 , u1 = -9632/375


## §4.2: eliminate accessories, then solve $E_j(t)=0$
Write $t=u_1\ne0$ and $\mathcal R=\Phi+\sum_k b_k e_k$, where the accessory
basis is $(1,(u-a_1)^{-1},\ldots)$. First set $L=t^2/H(u_0)$ and build the
necessary jets of $u(q)$ from
$$u''=(A'/A)u'+\tfrac L2A^2H'(u).$$
Let $J=(\frac32(A')^2-AA'')/A^4$. Its first $d$ derivatives determine the
$d$ accessories by a linear system; its remaining derivatives yield
$$E_j(t)=j![q^j]\bigl(L\mathcal R(u(q))-J(q)\bigr).$$
We check that the elimination determinant and all reduced denominators are
monomials in $t$, so no admissible nonzero tangent is lost. The polynomial gcd
of the remaining $E_j$ then proves uniqueness over $\mathbb C$.
The following cell prints the actual residual polynomials, not sampled values.

In [5]:
family = {
    '33a': (-384/(u-1)**2-108/u, [1]),
    '34a': (-QQ(1336336)/(16875*(u-QQ(16)/15)**2), [QQ(16)/15]),
    '46a': (-196608/(u-64)**2-3*u/8, [64]),
    '26a': (-QQ(507)/(2*u**2), [0]),
    '26b': (-3/u**2, [0]),
    '38a': (-QQ(57)/(2*u**2)-6*u, [0]),
    '38b': (-30/u**2, [0]),
    '58a': (-2523/u**2+QQ(65536)/(9*(u+27)**2), [0,-27]),
    '58b': (-3/u**2-QQ(65536)/(177147*(u+QQ(1)/27)**2), [0,-QQ(1)/27])}

for name, (Phi, poles) in family.items():
    H, a, c = data[name]; Phi = Fu(Phi)
    basis = [Fu(1)] + [1/(Fu(u)-p) for p in poles]
    d, last = len(basis), len(c)-3
    assert a not in poles and Phi.denominator()(a) != 0
    S = PowerSeriesRing(K, 'q', default_prec=len(c)+4)
    A = S([c[n]/factorial(n) for n in range(len(c))]).add_bigoh(len(c))
    J = (QQ(3)/2*A.derivative()**2-A*A.derivative(2))/A**4
    L = K(t**2/H(a)); jets = [K(a), K(t)]
    for n in range(last-1):
        U = S(jets)  # Rebuild an exact representative; do not reuse an O(q^k) truncation.
        rhs = A.derivative()/A*U.derivative() + L/2*A**2*H.derivative()(U)
        jets.append(K(rhs[n])/((n+1)*(n+2)))
    U = S(jets).add_bigoh(last+1)
    assert 2*jets[2] == t+L/2*H.derivative()(a)
    assert 6*jets[3] == c[2]*t+3*L/2*H.derivative()(a)+L/2*H.derivative(2)(a)*t
    fixed = at(Phi, U); B = [at(e, U) for e in basis]
    M = matrix(K, [[L*B[j][i] for j in range(d)] for i in range(d)])
    det = M.det()
    assert monomial(det.numerator()) and monomial(det.denominator())
    accessories = M.solve_right(vector(K, [J[i]-L*fixed[i] for i in range(d)]))
    assert all(monomial(f.denominator()) for f in [L]+list(accessories))
    residual = L*(fixed+sum(accessories[j]*B[j] for j in range(d)))-J
    assert all(residual[i] == 0 for i in range(d))
    E = {j: K(factorial(j)*residual[j]) for j in range(d,last+1)}
    assert E and all(e.denominator().degree() == 0 for e in E.values())
    common = gcd([Pt(e) for e in E.values()])
    assert common != 0
    while common[0] == 0:
        common = common//t
    common = common.monic()
    assert common.degree() == 1, (name, common)
    root = -common[0]; assert root != 0
    values = [QQ(at(f, root)) for f in [L]+list(accessories)]
    assert at(det, root) != 0 and all(at(e, root) == 0 for e in E.values())
    R = Fu(Phi+sum(values[j+1]*basis[j] for j in range(d)))
    assert root**2 == values[0]*H(a)
    recovered[name] = (R, values[0], a, root)
    fits[name] = dict(E=E, gcd=common, jets=jets, parameters=[L]+list(accessories))
    print(name, ': accessory parameters (C, B or B0, B1) =', values[1:])
    display(Math(r'\mathcal R_{'+name+r'}(u)='+latex(R)))
    for j, equation in E.items():
        display(Math(r'E_{'+str(j)+r'}^{'+name+r'}(t)='+latex(equation)))
    print('Unique admissible root:', root)

33a : accessory parameters (C, B or B0, B1) = [-4, -96]


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique admissible root: -19296/2875
34a : accessory parameters (C, B or B0, B1) = [31/50, 9248/1125]


<IPython.core.display.Math object>

... (28 outputs left out)

58b : accessory parameters (C, B or B0, B1) = [-39625/243, -34, -163840/6561]


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Unique admissible root: -16/33


## The displayed $26b$ example in §4.2 and the level-58 cases
The next cell checks the displayed $26b$ jets, elimination formulas (4.17),
and residual polynomials (4.18), including the unique common root $u_1=44$.
The preceding calculation also certifies all the other forms, including $33a$.
For $58b$, matching orders $0,1,2$ makes $E_2=0$ and the remaining $E_3$ is
linear. For $58a$, $E_3$ has five distinct nonzero roots; $E_4$, which uses
$c_6$, selects one. This explains the different input lengths.


In [6]:
f = fits['26b']; L, C, B = f['parameters']
assert 2*f['jets'][2] == t+QQ(67)/44*t**2
assert 6*f['jets'][3] == 40*t+QQ(201)/44*t**2+QQ(531)/176*t**3
assert L == -t**2/704
assert B == 6+681472/t**3
assert C == -3+27104/t**2-681472/t**3
assert f['E'][2] == -11616-462*t+QQ(3)/352*t**4
assert f['E'][3] == 2201232-1386*t-QQ(231)/2*t**2+QQ(9)/352*t**4-QQ(189)/15488*t**5
assert f['gcd'] == t-44
assert [at(p, 44) for p in (L, B, C)] == [-QQ(11)/4, 14, 3]
display(Math(r'\gcd(E_2^{26b},E_3^{26b})='+latex(f['gcd'])))
print('26b (displayed in Section 4.2): u1 = 44, L = -11/4, B = 14, C = 3.')
assert fits['58b']['E'][3] == QQ(85184)/81*(33*t+16)
p = Pt(fits['58a']['E'][3])
assert p.squarefree_part().degree() == 5 and p[0] != 0
print('PASS: the printed 26b formulas (4.17)-(4.18) and the two level-58 stopping criteria.')


<IPython.core.display.Math object>

26b (displayed in Section 4.2): u1 = 44, L = -11/4, B = 14, C = 3.
PASS: the printed 26b formulas (4.17)-(4.18) and the two level-58 stopping criteria.


## Recover the input coefficients from the solved systems
Starting afresh with only $A(0)=A'(0)=1$ and the solved $(\mathcal R,L,u_0,u_1)$,
we generate all input coefficients and one further coefficient. Both original
undivided equations and the first integral are checked to the available order.
The reference values in the last cell are comparisons, not solver inputs.

In [7]:
def coefficients(name, last):
    H = data[name][0]; R, L, a, b = recovered[name]
    S = PowerSeriesRing(QQ, 'q', default_prec=last+4)
    aa, uu = [QQ(1),QQ(1)], [a,b]
    for n in range(last-1):
        A, U = S(aa), S(uu)
        rhsA = QQ(3)/2*A.derivative()**2/A-L*at(R,U)*A**3
        rhsU = A.derivative()/A*U.derivative()+L/2*A**2*H.derivative()(U)
        aa.append(QQ(rhsA[n])/((n+1)*(n+2)))
        uu.append(QQ(rhsU[n])/((n+1)*(n+2)))
    A, U = S(aa), S(uu)
    rA = A*A.derivative(2)-QQ(3)/2*A.derivative()**2+L*at(R,U)*A**4
    rU = 2*A*U.derivative(2)-2*A.derivative()*U.derivative()-L*A**3*H.derivative()(U)
    first = U.derivative()**2-L*A**2*H(U)
    assert all(rA[j] == rU[j] == 0 for j in range(last-1))
    assert all(first[j] == 0 for j in range(last))
    return [factorial(n)*aa[n] for n in range(last+1)]

generated = {name: coefficients(name,len(c)) for name, (H,a,c) in data.items()}
assert all(generated[name][:-1] == c for name, (H,a,c) in data.items())

In [8]:
# Reference tuples (R, L, u1, first coefficient beyond the supplied range).
reference = {
    "14a": (Fu(3*(4-u)/8), rationals("-11/9 -352/3 24")),
    "15a": (Fu(-4-4*u/3), rationals("-7/64 -63 -381/64")),
    "21a": (Fu(-16*(17*u**2+392*u-2401)/(u-49)**2), rationals("-43/3969 -9632/375 -4357004/5250987")),
    "33a": (Fu(-4*(u**3+49*u**2+19*u+27)/(u*(u-1)**2)), rationals("-67/529 -19296/2875 176790432531276/148035889")),
    "34a": (Fu((279*u**2+3104*u-39264)/(2*(15*u-16)**2)), rationals("-11/25 -3179/80 6553872/15625")),
    "46a": (Fu(-(3*u**3+52*u**2+13824*u-311296)/(8*(u-64)**2)), rationals("-844992/61701025 -15188992/636255 939622156860406531404672/234896819381221764390625")),
    "26a": (Fu(9/2-9/u-507/(2*u**2)), rationals("-11/1764 44/21 5961776/28588707")),
    "26b": (Fu(3+14/u-3/u**2), rationals("-11/4 44 5134480")),
    "38a": (Fu(-6*u-45/2-57/u-57/(2*u**2)), rationals("-11/324 44/9 -38071696/177147")),
    "38b": (Fu(-16-52/u-30/u**2), rationals("-11/676 44/13 25568336/4826809")),
    "58a": (Fu(5-144358/(243*u)-2523/u**2-229376/(243*(u+27))+65536/(9*(u+27)**2)), rationals("121/400 -176/5 -4829523664/78125")),
    "58b": (Fu(-39625/243-34/u-3/u**2-163840/(243*(27*u+1))-65536/(243*(27*u+1)**2)), rationals("-11/144 -16/33 -303950320/729")),
}
rows = ['| Form | u0 | u1 | L | First predicted coefficient |', '|---|---:|---:|---:|---:|']
for name, (expected_R, expected) in reference.items():
    R, L, a, b = recovered[name]
    assert R == expected_R and [L,b,generated[name][-1]] == expected, name
    rows.append('| '+ ' | '.join(map(str,[name,a,b,L,generated[name][-1]]))+' |')
display(Markdown('\n'.join(rows)))
print('PASS: all twelve parameter recoveries, all supplied coefficients, and all twelve first predictions.')

| Form | u0 | u1 | L | First predicted coefficient |
|---|---:|---:|---:|---:|
| 14a | 16 | -352/3 | -11/9 | 24 |
| 15a | 9 | -63 | -7/64 | -381/64 |
| 21a | 441/25 | -9632/375 | -43/3969 | -4357004/5250987 |
| 33a | 9/25 | -19296/2875 | -67/529 | 176790432531276/148035889 |
| 34a | -15/4 | -3179/80 | -11/25 | 6553872/15625 |
| 46a | 3136/81 | -15188992/636255 | -844992/61701025 | 939622156860406531404672/234896819381221764390625 |
| 26a | 1 | 44/21 | -11/1764 | 5961776/28588707 |
| 26b | 1 | 44 | -11/4 | 5134480 |
| 38a | 1 | 44/9 | -11/324 | -38071696/177147 |
| 38b | 1 | 44/13 | -11/676 | 25568336/4826809 |
| 58a | -11 | -176/5 | 121/400 | -4829523664/78125 |
| 58b | -1/11 | -16/33 | -11/144 | -303950320/729 |

PASS: all twelve parameter recoveries, all supplied coefficients, and all twelve first predictions.


The calculations use exact rational functions, formal power series and polynomial gcds.
The general geometric arguments remain those of the paper. Tu's normalization and
Schwarzian inputs are from [Tu, §5.1](https://msp.org/pjm/2014/269-2/pjm-v269-n2-p14-s.pdf);
the coordinate changes are quoted and checked above. AI assistance and the GPL-3.0
license are described in the package README.